# **Limpieza de Datos**
## ATUS Anual 2024
### Proyecto Final — Almacenes y Minería de Datos

Este notebook aplica las decisiones de limpieza identificadas durante el análisis
exploratorio (`analisis_exploratorio.ipynb`). Cada decisión está justificada con
base en el diccionario oficial de INEGI y los hallazgos del EDA.

El notebook recibe como entrada el dataset original `atus_anual_2024.csv` y produce
como salida el dataset limpio `atus_anual_2024_limpio.csv`, que servirá como punto
de partida para el análisis estadístico, la clasificación supervisada y el
agrupamiento no supervisado.

In [2]:
import pandas as pd

dataset_raw = pd.read_csv("../data/atus_anual_2024.csv")

print(f"Dimensiones del dataset original: {dataset_raw.shape}")

Dimensiones del dataset original: (390627, 45)


## 1. Filas Duplicadas

En el análisis de calidad identificamos 871 filas con valores idénticos en todas
sus columnas (≈0.22% del dataset). Se inspeccionó su contenido para verificar que no correspondían a accidentes distintos con las mismas características.

La inspección confirmó que los registros duplicados coinciden en todas las variables
relevantes: ubicación exacta, momento del accidente (hasta el minuto), tipo de percance,
causa presunta, características del conductor y severidad. Que dos accidentes
independientes compartan simultáneamente todos estos atributos es estadísticamente
improbable, por lo que se concluye que se deben a errores de captura o registros
duplicados en la fuente.

**Decisión:** Se eliminan las 871 filas duplicadas conservando una única ocurrencia
de cada registro.

In [3]:
dataset_limpio = dataset_raw.drop_duplicates().copy()

print(f"Filas antes: {len(dataset_raw)}")
print(f"Filas eliminadas: {len(dataset_raw) - len(dataset_limpio)}")
print(f"Filas después: {len(dataset_limpio)}")
print(f"Duplicados restantes: {dataset_limpio.duplicated().sum()}")

Filas antes: 390627
Filas eliminadas: 871
Filas después: 389756
Duplicados restantes: 0


## 2. Columnas sin Variación

Durante el análisis de calidad identificamos cuatro columnas que contienen un solo
valor en todos sus registros: `COBERTURA`, `ANIO`, `NEMUERTO` y `NEHERIDO`. Dado que
no permiten distinguir entre accidentes ni aportan capacidad predictiva, serán
eliminadas.

La eliminación no implica que estas columnas sean incorrectas, `COBERTURA` indica
que todo el dataset está a nivel municipal y `ANIO` confirma que todos los registros
corresponden a 2024. Su información quedó documentada durante el EDA y no es necesario
conservarlas para las siguientes etapas.

In [4]:
columnas_sin_variacion = ["COBERTURA", "ANIO", "NEMUERTO", "NEHERIDO"]

dataset_limpio = dataset_limpio.drop(columns=columnas_sin_variacion)

print(f"Columnas eliminadas: {columnas_sin_variacion}")
print(f"Columnas restantes: {dataset_limpio.shape[1]}")

Columnas eliminadas: ['COBERTURA', 'ANIO', 'NEMUERTO', 'NEHERIDO']
Columnas restantes: 41


## 3. Registros con Certificado Cero

En la sección 4.4 del EDA identificamos 15,678 registros donde `TIPACCID` contiene
el valor `Certificado cero`. De acuerdo con la documentación oficial del INEGI, estos
registros corresponden a municipios que informaron formalmente la ausencia de accidentes
viales en su jurisdicción. No representan percances reales.

Antes de eliminarlos, inspeccionamos una muestra para verificar que el resto de sus
columnas tampoco contiene información útil sobre accidentes.

In [5]:
columnas_inspeccion = [
    "ID_ENTIDAD", "ID_MUNICIPIO", "MES", "ID_DIA", "DIASEMANA",
    "TIPACCID", "CAUSAACCI", "URBANA", "SEXO", "ID_EDAD",
    "AUTOMOVIL", "MOTOCICLET", "CONDMUERTO", "CONDHERIDO",
    "PASAMUERTO", "PASAHERIDO", "CLASACC"
]

muestra_certificado_cero = (
    dataset_limpio[dataset_limpio["TIPACCID"] == "Certificado cero"][columnas_inspeccion]
    .sample(10, random_state=42)
)

muestra_certificado_cero

,ID_ENTIDAD,ID_MUNICIPIO,MES,ID_DIA,DIASEMANA,TIPACCID,CAUSAACCI,URBANA,SEXO,ID_EDAD,AUTOMOVIL,MOTOCICLET,CONDMUERTO,CONDHERIDO,PASAMUERTO,PASAHERIDO,CLASACC
277266,20,405,12,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños
273704,20,185,8,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños
274646,20,263,7,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños
269881,20,52,4,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños
275738,20,335,1,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños
376277,30,171,9,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños
275273,20,304,10,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños
367470,30,17,8,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños
117373,12,40,8,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños
132142,14,40,3,28,lunes,Certificado cero,Certificado cero,Sin accidente en esta zona,Certificado cero,0,0,0,0,0,0,0,Sólo daños


La muestra confirma que todos los registros con `Certificado cero` presentan valores
genéricos: las variables de conteo de vehículos y víctimas son cero, `CLASACC` siempre
es `Sólo daños` y los campos del conductor contienen valores que tampoco representan
información real.

Conservar estos registros introduciría distorsiones concretas en el análisis:

- **Distribuciones temporales:** Inflarían artificialmente la frecuencia del día `28`
  y del valor `lunes` en `DIASEMANA`.
- **Variables de conteo:** Reforzarían el sesgo hacia cero en las columnas de vehículos
  y víctimas más allá de lo que los accidentes reales justifican.
- **Variable objetivo:** Añadirían 15,678 registros clasificados como `Sólo daños`
  sin ninguna base en un accidente real, agravando el desbalance de clases.
- **Agrupamiento:** Formarían un cluster artificial homogéneo sin interpretación útil
  sobre la siniestralidad.

**Decisión:** Se eliminan los 15,678 registros con `Certificado cero`. A partir de
este punto, cada fila de `dataset_limpio` corresponde a un accidente registrado.

In [6]:
cert_cero = (dataset_limpio["TIPACCID"] == "Certificado cero").sum()
dataset_limpio = dataset_limpio[dataset_limpio["TIPACCID"] != "Certificado cero"].copy()

print(f"Registros eliminados: {cert_cero}")
print(f"Filas restantes: {len(dataset_limpio)}")
print(f"Certificados cero restantes: {(dataset_limpio['TIPACCID'] == 'Certificado cero').sum()}")

Registros eliminados: 15678
Filas restantes: 374078
Certificados cero restantes: 0


## 4. Códigos Especiales en `ID_EDAD`

La columna `ID_EDAD` utiliza dos códigos especiales documentados en el diccionario
oficial: el valor `0` indica que el conductor se fugó y el valor `99` indica que se
desconoce su edad. Ninguno representa una edad real.

Aunque el dataset no reporta valores nulos, estos códigos son efectivamente **datos
faltantes enmascarados**. Tratarlos como valores numéricos distorsionaría estadísticas
de edad y afectaría negativamente los modelos.

Para preservar el significado de ambos códigos sin contaminar la variable numérica,
se crean dos columnas binarias auxiliares antes de reemplazarlos por `NaN`:

- `CONDUCTOR_FUGADO`: vale `1` cuando `ID_EDAD == 0`
- `EDAD_DESCONOCIDA`: vale `1` cuando `ID_EDAD == 99`

In [7]:
dataset_limpio["CONDUCTOR_FUGADO"] = (dataset_limpio["ID_EDAD"] == 0).astype(int)
dataset_limpio["EDAD_DESCONOCIDA"] = (dataset_limpio["ID_EDAD"] == 99).astype(int)

dataset_limpio["ID_EDAD"] = dataset_limpio["ID_EDAD"].replace({0: pd.NA, 99: pd.NA})

print(f"Registros con conductor fugado:  {dataset_limpio['CONDUCTOR_FUGADO'].sum():,}")
print(f"Registros con edad desconocida:  {dataset_limpio['EDAD_DESCONOCIDA'].sum():,}")
print(f"Edades válidas:                  {dataset_limpio['ID_EDAD'].notna().sum():,}")
print(f"Valores nulos en ID_EDAD:        {dataset_limpio['ID_EDAD'].isna().sum():,}")

Registros con conductor fugado:  34,920
Registros con edad desconocida:  61,302
Edades válidas:                  277,856
Valores nulos en ID_EDAD:        96,222


## 5. Eliminación de `ESTATUS`

La columna `ESTATUS` indica el estado de publicación de las cifras de INEGI
(`Cifras Definitivas` o `Cifras Corregidas`). Esta información describe la procedencia
administrativa de los registros, no una característica del accidente. Mantenerla
durante el modelado introduciría una variable sin relación con la severidad y durante
el agrupamiento podría influir en la formación de grupos sin aportar interpretación
útil.

**Decisión:** Se elimina `ESTATUS` del dataset.

In [8]:
dataset_limpio = dataset_limpio.drop(columns=["ESTATUS"])

print(f"Columnas restantes: {dataset_limpio.shape[1]}")

Columnas restantes: 42


## 6. Incorporación de Nombres de Entidades y Municipios

Las columnas `ID_ENTIDAD` e `ID_MUNICIPIO` almacenan claves numéricas del catálogo
oficial de INEGI. Para facilitar la interpretación de los resultados del análisis y
del modelado, incorporamos los nombres oficiales mediante un cruce con el Catálogo
Único de Claves de Áreas Geoestadísticas Estatales y Municipales (AGEEML), publicado
y mantenido por el propio INEGI.

Se descargó el catálogo de Áreas Geoestadísticas Municipales (AGEM) desde el portal de INEGI AGEEML, sin aplicar filtros de entidad y utilizando el nivel de desagregación Área Geoestadística Municipal (AGEM).

El archivo fue almacenado localmente en la ruta: `data/AGEEML.csv`

In [9]:
catalogo = pd.read_csv(
    "../data/AGEEML.csv",
    dtype=str,
    encoding="latin-1",
    usecols=["CVE_ENT", "NOM_ENT", "CVE_MUN", "NOM_MUN"]
)

catalogo["CVE_ENT"] = catalogo["CVE_ENT"].astype(int)
catalogo["CVE_MUN"] = catalogo["CVE_MUN"].astype(int)

print(f"Entidades en el catálogo: {catalogo['CVE_ENT'].nunique()}")
print(f"Municipios en el catálogo: {len(catalogo)}")
catalogo.head()

Entidades en el catálogo: 32
Municipios en el catálogo: 2478


,CVE_ENT,NOM_ENT,CVE_MUN,NOM_MUN
0,1,Aguascalientes,1,Aguascalientes
1,1,Aguascalientes,2,Asientos
2,1,Aguascalientes,3,Calvillo
3,1,Aguascalientes,4,Cosío
4,1,Aguascalientes,5,Jesús María


In [10]:
dataset_limpio = (
    dataset_limpio
    .merge(
        catalogo,
        left_on=["ID_ENTIDAD", "ID_MUNICIPIO"],
        right_on=["CVE_ENT", "CVE_MUN"],
        how="left"
    )
    .drop(columns=["ID_ENTIDAD", "ID_MUNICIPIO", "CVE_ENT", "CVE_MUN"])
)

sin_nombre = dataset_limpio["NOM_ENT"].isna().sum()
print(f"Registros sin nombre de entidad tras el cruce: {sin_nombre}")
print(f"Columnas agregadas: NOM_ENT, NOM_MUN")
print(f"Dimensiones: {dataset_limpio.shape}")

Registros sin nombre de entidad tras el cruce: 0
Columnas agregadas: NOM_ENT, NOM_MUN
Dimensiones: (374078, 42)


Verificamos que no existan registros donde `NOM_ENT` y `NOM_MUN` tengan el mismo
valor por error.

In [11]:
mismos_valores = (dataset_limpio["NOM_ENT"] == dataset_limpio["NOM_MUN"]).sum()
print(f"Registros donde NOM_ENT == NOM_MUN: {mismos_valores}")

if mismos_valores > 0:
    display(
        dataset_limpio[dataset_limpio["NOM_ENT"] == dataset_limpio["NOM_MUN"]]
        [["NOM_ENT", "NOM_MUN"]]
        .value_counts()
        .reset_index()
        .rename(columns={"count": "Cantidad de registros"})
    )

Registros donde NOM_ENT == NOM_MUN: 37022


,NOM_ENT,NOM_MUN,Cantidad de registros
0,Chihuahua,Chihuahua,9724
1,Puebla,Puebla,6828
2,Durango,Durango,6268
3,Querétaro,Querétaro,3583
4,San Luis Potosí,San Luis Potosí,3370
5,Aguascalientes,Aguascalientes,3213
6,Colima,Colima,1285
7,Campeche,Campeche,1242
8,Zacatecas,Zacatecas,645
9,Tlaxcala,Tlaxcala,430


Los casos donde `NOM_ENT == NOM_MUN` corresponden a municipios cabecera que comparten
nombre con su entidad federativa (Chihuahua-Chihuahua, Puebla-Puebla, etc.). No
representan un error en el catálogo ni en el cruce.

Las columnas `NOM_ENT` y `NOM_MUN` reemplazan a `ID_ENTIDAD` e `ID_MUNICIPIO` con
los nombres oficiales. Los identificadores numéricos se eliminan al ser información
redundante.

## 7. Corrección de `DIASEMANA`

La columna `DIASEMANA` presenta inconsistencias en su formato: `lunes` está en
minúsculas mientras el resto de los valores están capitalizados.

In [12]:
mapa_diasemana = {
    'lunes':     'Lunes',
    'Martes':    'Martes',
    'Miercoles': 'Miercoles',
    'Jueves':    'Jueves',
    'Viernes':   'Viernes',
    'Sabado':    'Sabado',
    'Domingo':   'Domingo'
}

dataset_limpio['DIASEMANA'] = dataset_limpio['DIASEMANA'].replace(mapa_diasemana)

print("Valores únicos tras la corrección:")
print(dataset_limpio['DIASEMANA'].unique())

Valores únicos tras la corrección:
<StringArray>
['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']
Length: 7, dtype: str


## 8. Corrección de `SEXO`

Durante el análisis de calidad verificamos que `SEXO` utiliza el valor `'Se fugó'`
para los mismos 34,920 registros donde `CONDUCTOR_FUGADO == 1`. Ambas columnas
capturan exactamente el mismo fenómeno, confirmando colinealidad perfecta.

El valor `'Se fugó'` mezcla dos conceptos distintos: género e identidad del conductor.
Se reemplaza por `'Desconocido'` ya que efectivamente no se conoce el sexo del
conductor que abandonó la escena, mantiendo consistencia con el tratamiento de valores
no disponibles en otras columnas del dataset (como `'Se ignora'` en `CINTURON`).
La información de fuga queda correctamente representada por `CONDUCTOR_FUGADO`.

In [13]:
dataset_limpio['SEXO'] = dataset_limpio['SEXO'].replace('Se fugó', 'Desconocido')

print("Valores únicos en SEXO tras la corrección:")
print(dataset_limpio['SEXO'].value_counts())

Valores únicos en SEXO tras la corrección:
SEXO
Hombre         277364
Mujer           61794
Desconocido     34920
Name: count, dtype: int64


## 9. Resumen del Proceso de Limpieza

In [14]:
print("=" * 60)
print("         RESUMEN DEL PROCESO DE LIMPIEZA")
print("=" * 60)

print(f"\n{'DIMENSIONES':}")
print(f"  Dataset original:  {dataset_raw.shape[0]:>7,} filas × {dataset_raw.shape[1]} columnas")
print(f"  Dataset limpio:    {dataset_limpio.shape[0]:>7,} filas × {dataset_limpio.shape[1]} columnas")

print(f"\n{'FILAS ELIMINADAS':}")
duplicados_n  = dataset_raw.shape[0] - dataset_raw.drop_duplicates().shape[0]
cert_cero_n   = (dataset_raw['TIPACCID'] == 'Certificado cero').sum()
print(f"  Filas duplicadas:            {duplicados_n:>6,}")
print(f"  Registros Certificado cero:  {cert_cero_n:>6,}")
print(f"  Total eliminadas:            {dataset_raw.shape[0] - dataset_limpio.shape[0]:>6,}")

print(f"\n{'COLUMNAS ELIMINADAS':}")
cols_eliminadas = [
    ('COBERTURA',    'Sin variación'),
    ('ANIO',         'Sin variación'),
    ('NEMUERTO',     'Sin variación'),
    ('NEHERIDO',     'Sin variación'),
    ('ESTATUS',      'Variable administrativa irrelevante'),
    ('ID_ENTIDAD',   'Reemplazada por NOM_ENT'),
    ('ID_MUNICIPIO', 'Reemplazada por NOM_MUN'),
]
for col, motivo in cols_eliminadas:
    print(f"  {col:<20} — {motivo}")

print(f"\n{'COLUMNAS AGREGADAS':}")
cols_agregadas = [
    ('NOM_ENT',          'Nombre oficial de la entidad federativa (AGEEML)'),
    ('NOM_MUN',          'Nombre oficial del municipio (AGEEML)'),
    ('CONDUCTOR_FUGADO', 'Bandera binaria: conductor abandonó la escena'),
    ('EDAD_DESCONOCIDA', 'Bandera binaria: edad del conductor desconocida'),
]
for col, motivo in cols_agregadas:
    print(f"  {col:<20} — {motivo}")

print(f"\n{'COLUMNAS MODIFICADAS':}")
cols_modificadas = [
    ('ID_EDAD',   'Valores 0 y 99 reemplazados por NaN'),
    ('DIASEMANA', 'Capitalización y tildes corregidas'),
    ('SEXO',      '"Se fugó" reemplazado por "Desconocido"'),
]
for col, motivo in cols_modificadas:
    print(f"  {col:<20} — {motivo}")

print(f"\n{'DECISIONES SIN MODIFICACIÓN':}")
print(f"  CINTURON             — Se conservan las tres categorías (Sí / No / Se ignora)")

print("\n" + "=" * 60)
print(f"  Dataset final listo: {dataset_limpio.shape[0]:,} filas × {dataset_limpio.shape[1]} columnas")
print("=" * 60)

         RESUMEN DEL PROCESO DE LIMPIEZA

DIMENSIONES
  Dataset original:  390,627 filas × 45 columnas
  Dataset limpio:    374,078 filas × 42 columnas

FILAS ELIMINADAS
  Filas duplicadas:               871
  Registros Certificado cero:  15,678
  Total eliminadas:            16,549

COLUMNAS ELIMINADAS
  COBERTURA            — Sin variación
  ANIO                 — Sin variación
  NEMUERTO             — Sin variación
  NEHERIDO             — Sin variación
  ESTATUS              — Variable administrativa irrelevante
  ID_ENTIDAD           — Reemplazada por NOM_ENT
  ID_MUNICIPIO         — Reemplazada por NOM_MUN

COLUMNAS AGREGADAS
  NOM_ENT              — Nombre oficial de la entidad federativa (AGEEML)
  NOM_MUN              — Nombre oficial del municipio (AGEEML)
  CONDUCTOR_FUGADO     — Bandera binaria: conductor abandonó la escena
  EDAD_DESCONOCIDA     — Bandera binaria: edad del conductor desconocida

COLUMNAS MODIFICADAS
  ID_EDAD              — Valores 0 y 99 reemplazados por 

## 10. Exportación del Dataset Limpio

In [15]:
ruta_dataset_limpio = "../data/atus_anual_2024_limpio.csv"
dataset_limpio.to_csv(ruta_dataset_limpio, index=False)
print(f"Dataset limpio exportado en: {ruta_dataset_limpio}")
print(f"Dimensiones finales: {dataset_limpio.shape[0]:,} filas × {dataset_limpio.shape[1]} columnas")

Dataset limpio exportado en: ../data/atus_anual_2024_limpio.csv
Dimensiones finales: 374,078 filas × 42 columnas


El archivo `atus_anual_2024_limpio.csv` contiene 374,078 filas y 42 columnas. Cada
fila corresponde a un accidente registrado, sin duplicados exactos, con nombres
oficiales de entidad y municipio, y con los códigos especiales correctamente tratados.
Este dataset servirá como base para el análisis estadístico descriptivo, la
clasificación supervisada y el agrupamiento no supervisado.